<a href="https://colab.research.google.com/github/Baptistecaille/snn_pc_hybrid/blob/main/full_gpu_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SNN-PC Hybrid Full Training on Massive GPU

This notebook is dedicated to long-running full-curriculum training on a large CUDA GPU.

What it is optimized for:
- end-to-end curriculum from `bootstrap` to `oscar_full`
- CUDA-first execution with strict GPU validation
- large dataset buffers and automatic checkpoint resume
- separate output directories from the laptop notebook
- lightweight monitoring for checkpoints, logs, and plots

## 1. Environment Check

This section validates that the notebook is attached to a CUDA-capable runtime before doing any heavy work.

In [1]:
!git clone https://github.com/Baptistecaille/snn_pc_hybrid
%cd snn_pc_hybrid
!pip install -r requirements.txt

fatal: destination path 'snn_pc_hybrid' already exists and is not an empty directory.
/content/snn_pc_hybrid


In [2]:
import platform
import sys

try:
    import torch
except ImportError:
    torch = None

print({
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'torch_installed': torch is not None,
})

if torch is None:
    raise ImportError('Torch is not installed in the active notebook environment.')

print({
    'torch_version': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_device_count': torch.cuda.device_count() if torch.cuda.is_available() else 0,
})

if torch.cuda.is_available():
    for device_index in range(torch.cuda.device_count()):
        print({
            'device_index': device_index,
            'device_name': torch.cuda.get_device_name(device_index),
            'device_capability': torch.cuda.get_device_capability(device_index),
        })

{'python': '3.12.12', 'platform': 'Linux-6.6.113+-x86_64-with-glibc2.35', 'torch_installed': True}
{'torch_version': '2.10.0+cu128', 'cuda_available': True, 'cuda_device_count': 1}
{'device_index': 0, 'device_name': 'Tesla T4', 'device_capability': (7, 5)}


## 2. Locate Repository and Install Dependencies

This section locates the repository and installs missing packages into the active kernel when needed.

In [3]:
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path
import random
import numpy as np

# User provided path
REPO_DIR = Path('/content/snn_pc_hybrid')

if not REPO_DIR.exists():
    raise FileNotFoundError(f'Could not locate the repository at {REPO_DIR}')

def has_module(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None

def run_command(command: list[str], description: str) -> None:
    print(f'Running: {description}')
    subprocess.run(command, check=True)

def ensure_pip_available() -> bool:
    if has_module('pip'):
        return True
    try:
        run_command([sys.executable, '-m', 'ensurepip', '--upgrade'], 'bootstrap pip with ensurepip')
    except (subprocess.CalledProcessError, FileNotFoundError):
        return has_module('pip')
    return has_module('pip')

def install_with_pip(requirement_args: list[str]) -> None:
    if not ensure_pip_available():
        raise RuntimeError('pip is not available in the active Python environment.')
    run_command([sys.executable, '-m', 'pip', 'install', *requirement_args], f'pip install {requirement_args}')

def install_with_uv(requirement_args: list[str]) -> None:
    uv_executable = shutil.which('uv')
    if not uv_executable:
        raise RuntimeError('uv is not installed or not available on PATH.')
    run_command([uv_executable, 'pip', 'install', '--python', sys.executable, *requirement_args], f'uv pip install {requirement_args}')

def install_requirements(requirement_args: list[str], installer: str) -> None:
    preferred_installer = installer.lower().strip()
    if preferred_installer == 'uv':
        try:
            install_with_uv(requirement_args)
            return
        except RuntimeError as exc:
            print(f'uv install unavailable: {exc}. Falling back to pip.')
    try:
        install_with_pip(requirement_args)
    except RuntimeError as exc:
        print(f'pip install unavailable: {exc}')
        install_with_uv(requirement_args)

INSTALL_REQUIREMENTS = False
AUTO_INSTALL_MISSING_DEPS = True
PREFERRED_INSTALLER = 'pip'

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.append(str(REPO_DIR))

print({'repo_dir': str(REPO_DIR), 'python_executable': sys.executable})

required_modules = {
    'transformers': 'transformers>=4.38',
    'sentencepiece': 'sentencepiece>=0.1.99',
    'datasets': 'datasets>=2.18',
    'pandas': 'pandas>=2.1',
    'matplotlib': 'matplotlib>=3.8',
    'seaborn': 'seaborn>=0.13',
}
missing_requirements = [requirement for module_name, requirement in required_modules.items() if not has_module(module_name)]
print({'missing_requirements': missing_requirements})

if INSTALL_REQUIREMENTS:
    install_requirements(['--upgrade', 'pip', 'setuptools', 'wheel'], PREFERRED_INSTALLER)
    install_requirements(['-r', 'requirements.txt'], PREFERRED_INSTALLER)
    install_requirements(['pandas', 'matplotlib', 'seaborn'], PREFERRED_INSTALLER)
elif AUTO_INSTALL_MISSING_DEPS and missing_requirements:
    install_requirements(missing_requirements, PREFERRED_INSTALLER)

print('Notebook dependencies and repository path are configured.')

{'repo_dir': '/content/snn_pc_hybrid', 'python_executable': '/usr/bin/python3'}
{'missing_requirements': []}
Notebook dependencies and repository path are configured.


## 3. Massive GPU Training Preset

This section configures a full-curriculum run tuned for a large CUDA machine.

In [ ]:
SEED = 42
TRAINING_PROFILE = 'massive_gpu_full'
REQUIRE_CUDA = True
RUN_DATA_PREVIEW = False
AUTO_RESUME_LATEST = True
EXPLICIT_RESUME_PHASE = None  # Example: 'wikipedia_long'
START_PHASE = 'bootstrap'
END_PHASE = 'oscar_full'
MAX_STEPS_OVERRIDE = None
n_articles = 10000
WIKI_MAX_ARTICLES = None
OSCAR_BUFFER_SIZE = 50000
BATCH_SIZE_HINT = 32
LEARNING_RATE_HINT = 1e-3

if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError('This notebook expects a CUDA GPU runtime.')

print({
    'training_profile': TRAINING_PROFILE,
    'require_cuda': REQUIRE_CUDA,
    'auto_resume_latest': AUTO_RESUME_LATEST,
    'explicit_resume_phase': EXPLICIT_RESUME_PHASE,
    'start_phase': START_PHASE,
    'end_phase': END_PHASE,
    'max_steps_override': MAX_STEPS_OVERRIDE,
    'n_articles': n_articles,
    'wiki_max_articles': WIKI_MAX_ARTICLES,
    'oscar_buffer_size': OSCAR_BUFFER_SIZE,
    'batch_size_hint': BATCH_SIZE_HINT,
    'learning_rate_hint': LEARNING_RATE_HINT,
})

{'training_profile': 'massive_gpu_full', 'require_cuda': True, 'auto_resume_latest': True, 'explicit_resume_phase': None, 'start_phase': 'bootstrap', 'end_phase': 'oscar_full', 'max_steps_override': None, 'wiki_max_articles': None, 'oscar_buffer_size': 50000, 'batch_size_hint': 32, 'learning_rate_hint': 0.001}


## 4. Configure Paths and CUDA Runtime

This section sets separate output directories for the GPU run and enables CUDA-friendly runtime defaults.

In [5]:
import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUTPUT_ROOT = REPO_DIR / 'gpu_runs' / TRAINING_PROFILE
RESULTS_DIR = OUTPUT_ROOT / 'results'
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
LOG_CSV = OUTPUT_ROOT / 'logs' / 'training_log.csv'
HF_HOME = REPO_DIR / 'cache'

for path in [RESULTS_DIR, CHECKPOINT_DIR, LOG_CSV.parent, HF_HOME]:
    path.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_HOME)
os.environ['TRANSFORMERS_CACHE'] = str(HF_HOME / 'transformers')
os.environ['HF_DATASETS_CACHE'] = str(HF_HOME / 'datasets')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    torch.set_float32_matmul_precision('high')
    torch.backends.cudnn.benchmark = True
    if hasattr(torch.backends.cuda.matmul, 'allow_tf32'):
        torch.backends.cuda.matmul.allow_tf32 = True
    if hasattr(torch.backends.cudnn, 'allow_tf32'):
        torch.backends.cudnn.allow_tf32 = True

print({
    'device': device,
    'output_root': str(OUTPUT_ROOT),
    'results_dir': str(RESULTS_DIR),
    'checkpoint_dir': str(CHECKPOINT_DIR),
    'log_csv': str(LOG_CSV),
    'hf_home': str(HF_HOME),
})

{'device': 'cuda', 'output_root': '/content/snn_pc_hybrid/gpu_runs/massive_gpu_full', 'results_dir': '/content/snn_pc_hybrid/gpu_runs/massive_gpu_full/results', 'checkpoint_dir': '/content/snn_pc_hybrid/gpu_runs/massive_gpu_full/checkpoints', 'log_csv': '/content/snn_pc_hybrid/gpu_runs/massive_gpu_full/logs/training_log.csv', 'hf_home': '/content/snn_pc_hybrid/cache'}


## 5. Optional Data Preview

Keep this disabled for the full run unless you want a quick tokenizer and dataset sanity check.

In [6]:
# PATCH: redirect 'wikipedia' -> 'wikimedia/wikipedia' for datasets >= 2.17
import datasets as _datasets
_original_load = _datasets.load_dataset

def _patched_load(path, *args, **kwargs):
    if path == "wikipedia":
        path = "wikimedia/wikipedia"
        kwargs.pop("trust_remote_code", None)
    return _original_load(path, *args, **kwargs)

_datasets.load_dataset = _patched_load
print("Patch applied: 'wikipedia' -> 'wikimedia/wikipedia'")

Patch applied: 'wikipedia' -> 'wikimedia/wikipedia'


In [7]:
from torch.utils.data import DataLoader, random_split
import importlib

import config as config_module
import run_training as run_training_module
import training.datasets as datasets_module

config_module = importlib.reload(config_module)
run_training_module = importlib.reload(run_training_module)
datasets_module = importlib.reload(datasets_module)

SNNConfig = config_module.SNNConfig
load_tokenizer = run_training_module.load_tokenizer
WikiFrDataset = datasets_module.WikiFrDataset

preview_config = SNNConfig()
preview_config.data_cache_dir = str(HF_HOME)
preview_config.wiki_max_articles = 64
preview_config.phase_max_tokens['bootstrap'] = 16

tokenizer = load_tokenizer(preview_config)
validation_loader = None

if RUN_DATA_PREVIEW:
    preview_dataset = WikiFrDataset(
        tokenizer=tokenizer,
        max_tokens=preview_config.phase_max_tokens['bootstrap'],
        min_tokens=8,
        length_curriculum=True,
        cache_dir=str(HF_HOME / 'wikipedia'),
        config_name=preview_config.wikipedia_config_name,
        max_articles=preview_config.wiki_max_articles,
    )
    preview_len = len(preview_dataset)
    val_len = max(1, int(0.1 * preview_len))
    train_len = max(1, preview_len - val_len)
    preview_train, preview_val = random_split(
        preview_dataset,
        [train_len, val_len],
        generator=torch.Generator().manual_seed(SEED),
    )
    preview_loader = DataLoader(preview_train, batch_size=min(8, max(1, len(preview_train))), shuffle=False, num_workers=0, drop_last=False)
    validation_loader = DataLoader(preview_val, batch_size=min(8, max(1, len(preview_val))), shuffle=False, num_workers=0, drop_last=False)
    preview_batch = next(iter(preview_loader))
    print({
        'preview_dataset_size': preview_len,
        'train_preview_size': len(preview_train),
        'validation_preview_size': len(preview_val),
        'preview_batch_shape': tuple(preview_batch['input_ids'].shape),
    })
else:
    print('Data preview skipped for the massive GPU full-training workflow.')

  Chargement du tokenizer 'camembert-base'...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  Taille du vocabulaire : 32005
Data preview skipped for the massive GPU full-training workflow.


## 6. Build Full Training Configuration

This cell builds the runtime configuration used by the curriculum trainer.

In [8]:
train_config = SNNConfig()
train_config.data_cache_dir = str(HF_HOME)
train_config.checkpoint_dir = str(CHECKPOINT_DIR)
train_config.log_csv_path = str(LOG_CSV)
train_config.wiki_max_articles = WIKI_MAX_ARTICLES
train_config.oscar_buffer_size = OSCAR_BUFFER_SIZE
train_config.max_steps_override = MAX_STEPS_OVERRIDE
train_config.device = torch.device(device)

training_plan = {
    'training_profile': TRAINING_PROFILE,
    'start_phase': START_PHASE,
    'end_phase': END_PHASE,
    'max_steps_override': train_config.max_steps_override,
    'wiki_max_articles': train_config.wiki_max_articles,
    'oscar_buffer_size': train_config.oscar_buffer_size,
    'checkpoint_dir': train_config.checkpoint_dir,
    'log_csv_path': train_config.log_csv_path,
    'device': str(train_config.device),
}
training_plan

{'training_profile': 'massive_gpu_full',
 'start_phase': 'bootstrap',
 'end_phase': 'oscar_full',
 'max_steps_override': None,
 'wiki_max_articles': None,
 'oscar_buffer_size': 50000,
 'checkpoint_dir': '/content/snn_pc_hybrid/gpu_runs/massive_gpu_full/checkpoints',
 'log_csv_path': '/content/snn_pc_hybrid/gpu_runs/massive_gpu_full/logs/training_log.csv',
 'device': 'cuda'}

## 7. Initialize Model and Trainer

This section reloads the project modules and constructs a fresh trainer bound to the GPU configuration.

In [9]:
import importlib

import core.neuron as neuron_module
import modules.arcuate as arcuate_module
import modules.broca as broca_module
import modules.wernicke as wernicke_module
import run_training as run_training_module
import training.loss as loss_module
import training.trainer as trainer_module

neuron_module = importlib.reload(neuron_module)
wernicke_module = importlib.reload(wernicke_module)
broca_module = importlib.reload(broca_module)
arcuate_module = importlib.reload(arcuate_module)
run_training_module = importlib.reload(run_training_module)
loss_module = importlib.reload(loss_module)
trainer_module = importlib.reload(trainer_module)

build_model = run_training_module.build_model
FreeEnergyLoss = loss_module.FreeEnergyLoss
CurriculumTrainer = trainer_module.CurriculumTrainer

wernicke, broca, arcuate = build_model(train_config, tokenizer.vocab_size)
loss_fn = FreeEnergyLoss()
trainer = CurriculumTrainer(
    wernicke=wernicke,
    broca=broca,
    arcuate=arcuate,
    tokenizer=tokenizer,
    config=train_config,
)

print({
    'model_device': str(train_config.device),
    'checkpoint_dir': train_config.checkpoint_dir,
    'log_csv': train_config.log_csv_path,
})

  Paramètres totaux : 12,421,317
{'model_device': 'cuda', 'checkpoint_dir': '/content/snn_pc_hybrid/gpu_runs/massive_gpu_full/checkpoints', 'log_csv': '/content/snn_pc_hybrid/gpu_runs/massive_gpu_full/logs/training_log.csv'}


## 8. Resolve Resume Strategy

This section decides whether to start from scratch, resume a specific checkpoint, or continue from the latest completed phase.

In [10]:
PHASE_ORDER = ['bootstrap', 'wikipedia_short', 'wikipedia_long', 'oscar_filtered', 'oscar_full']
available_checkpoints = sorted(CHECKPOINT_DIR.glob('*.pt'))
available_checkpoint_phases = [path.stem for path in available_checkpoints]
latest_checkpoint_phase = available_checkpoint_phases[-1] if available_checkpoint_phases else None

resume_phase = EXPLICIT_RESUME_PHASE
effective_start_phase = START_PHASE

if resume_phase is None and AUTO_RESUME_LATEST and latest_checkpoint_phase is not None:
    resume_phase = latest_checkpoint_phase

if resume_phase is not None:
    resume_index = PHASE_ORDER.index(resume_phase)
    end_index = PHASE_ORDER.index(END_PHASE)
    if resume_index >= end_index:
        effective_start_phase = None
    else:
        effective_start_phase = PHASE_ORDER[resume_index + 1]

resume_plan = {
    'available_checkpoint_phases': available_checkpoint_phases,
    'latest_checkpoint_phase': latest_checkpoint_phase,
    'resume_phase': resume_phase,
    'effective_start_phase': effective_start_phase,
    'target_end_phase': END_PHASE,
}
resume_plan

{'available_checkpoint_phases': [],
 'latest_checkpoint_phase': None,
 'resume_phase': None,
 'effective_start_phase': 'bootstrap',
 'target_end_phase': 'oscar_full'}

## 9. Run Full Curriculum Training

This is the main long-running cell. It resumes from the chosen checkpoint strategy and trains until `oscar_full` unless the curriculum is already complete.

In [ ]:
import time

# ============================================================
#  LIMITE DU DATASET POUR TRAINING RAPIDE
#  Remplace n_articles par le nombre voulu :
#    - 10_000  → ~5 min de chunking, bon pour tester
#    - 50_000  → ~20 min, meilleure couverture
#    - 200_000 → ~1h30, training sérieux
# ============================================================

n_articles = 10_000

if hasattr(trainer, 'dataset'):
    ds = trainer.dataset
    if hasattr(ds, 'select'):
        trainer.dataset = ds.select(range(min(n_articles, len(ds))))
    elif isinstance(ds, dict) and 'train' in ds:
        trainer.dataset = ds['train'].select(range(min(n_articles, len(ds['train']))))
    print(f"✅ Dataset limité à {len(trainer.dataset):,} articles")
elif hasattr(trainer, 'train_dataset'):
    ds = trainer.train_dataset
    trainer.train_dataset = ds.select(range(min(n_articles, len(ds))))
    print(f"✅ Dataset limité à {len(trainer.train_dataset):,} articles")
else:
    print("⚠️ Impossible de trouver le dataset dans trainer — limite non appliquée")

# ============================================================

results = None
train_started_at = time.time()

if resume_phase is not None:
    trainer.load_checkpoint(resume_phase)
    print(f'Resumed weights from phase: {resume_phase}')

if effective_start_phase is None:
    print('The requested curriculum is already complete up to the selected end phase.')
    results = {}
else:
    print(f'Starting training: {effective_start_phase} -> {END_PHASE}')
    results = trainer.run(start_phase=effective_start_phase, end_phase=END_PHASE)

print('Training finished in seconds:', round(time.time() - train_started_at, 1))
results


## 10. Inspect Checkpoints and Logs

Use these cells to verify which phases completed and to inspect the tail of the CSV log.

In [ ]:
available_checkpoints = sorted(CHECKPOINT_DIR.glob('*.pt'))
[str(path) for path in available_checkpoints]

In [ ]:
import pandas as pd

if LOG_CSV.exists():
    log_df = pd.read_csv(LOG_CSV)
    display(log_df.tail(20))
else:
    print(f'Log file not found: {LOG_CSV}')

## 11. Post-Training Inference Sanity Check

Run this only after at least one later curriculum phase checkpoint exists. The output becomes meaningful after `wikipedia_long` and later.

In [ ]:
from core.oscillator import OscillatoryClock

sample_text = 'Le chat mange la souris.'
encoded = tokenizer(sample_text, return_tensors='pt', truncation=True, padding=False)
input_ids = encoded['input_ids'].to(train_config.device)
attention_mask = encoded['attention_mask'].to(train_config.device)

wernicke.reset_state()
broca.reset_state()
arcuate.reset_state()

with torch.no_grad():
    x_input = trainer._token_ids_to_bow(input_ids, attention_mask)
    inference_clock = OscillatoryClock(train_config)
    mu_prior = torch.zeros(x_input.shape[0], train_config.dim_wernicke, device=train_config.device)
    out_w = wernicke(x_input=x_input, mu_prior=mu_prior, clock=inference_clock)
    msg_w2b, _ = arcuate.transmit(message=out_w['prediction'], direction='W2B', visit_history=['W'])
    context = torch.zeros(x_input.shape[0], train_config.dim_broca, device=train_config.device)
    out_b = broca(mu_wernicke=msg_w2b, x_context=context, clock=inference_clock)
    top_probs, top_ids_tensor = out_b['logits'][0].softmax(dim=-1).topk(10)
    top_ids = top_ids_tensor.tolist()
    top_tokens = tokenizer.convert_ids_to_tokens(top_ids)

{
    'sample_text': sample_text,
    'broca_state': out_b['state'],
    'top_token_predictions': top_tokens,
    'top_token_probabilities': [round(float(prob), 6) for prob in top_probs],
}

## 12. Plot Logs and Generated Figures

This section displays recent PNG outputs and plots the main training metrics from the CSV log.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display

png_files = sorted(RESULTS_DIR.rglob('*.png'))
print(f'Found {len(png_files)} PNG files under {RESULTS_DIR}')
for png_path in png_files:
    print(png_path)
    display(Image(filename=str(png_path)))

if LOG_CSV.exists():
    log_df = pd.read_csv(LOG_CSV)
    if not log_df.empty:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(log_df['step'], log_df['F_total'])
        axes[0].set_title('Training Free Energy')
        axes[0].set_xlabel('Step')
        axes[0].set_ylabel('F_total')
        axes[1].plot(log_df['step'], log_df['r_W'], label='r_W')
        axes[1].plot(log_df['step'], log_df['r_B'], label='r_B')
        axes[1].set_title('Kuramoto Order Parameters')
        axes[1].set_xlabel('Step')
        axes[1].legend()
        plt.show()